# Estimación de la Función de Desempeño de Seguridad (FDS) para siniestralidad peatonal en Bogotá
### Modelos de conteo (Poisson / Binomial Negativa / ZINB) a nivel de segmento vial, con validación cruzada aleatoria de K = 5 pliegues

**Autor:** Brayan Hernandez - Santiago Toro — Tesis de grado, Ciencia de Datos

## 1. Entorno y reproducibilidad

In [1]:
## Librerías
import sys
import warnings
import numpy as np
import pandas as pd
import statsmodels
import sklearn
import statsmodels.api as sm
from scipy import stats
from scipy.special import expit
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error, mean_squared_error
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP

pd.set_option('display.max_columns', 80)

In [2]:
## Semilla y versiones del entorno (se documentan en la metodología)
SEMILLA = 1949
K_PLIEGUES = 5

print(f"Python: {sys.version.split()[0]}")
print(f"statsmodels: {statsmodels.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")

Python: 3.13.7
statsmodels: 0.14.6
scikit-learn: 1.9.0
pandas: 3.0.5


## 2. Datos

In [3]:
## Carga del conjunto unificado; el índice queda 0..n-1 porque las particiones trabajan por posición
df_unificado = pd.read_csv('Data/processed/segmentos_unificado.csv').reset_index(drop=True)

In [4]:
## Verificación de columnas y de la variable respuesta
COL_DIST = ['dist_puente_peatonal_m', 'dist_puente_vehicular_m', 'dist_semaforo_m', 'dist_peatonal_m']
COLS_REQ = COL_DIST + ['MVINUMC', 'MVI_VELREG', 'long_seg_m', 'n_siniestros']

faltan = [c for c in COLS_REQ if c not in df_unificado.columns]
assert not faltan, f'Faltan columnas: {faltan}'
assert df_unificado[COLS_REQ].notna().all().all(), 'Hay valores faltantes en las columnas del modelo'
assert (df_unificado['long_seg_m'] > 0).all(), 'Hay segmentos con longitud no positiva'

print(f"Segmentos: {len(df_unificado)} | % con cero siniestros: {100 * (df_unificado['n_siniestros'] == 0).mean():.2f}")
print(df_unificado['n_siniestros'].value_counts().sort_index().head(8).to_string())

Segmentos: 131705 | % con cero siniestros: 89.88
n_siniestros
0    118382
1      9827
2      2062
3       730
4       331
5       147
6        85
7        41


## 3. Offset y covariables

In [14]:
## Respuesta y offset: el logaritmo de la longitud es el único término de exposición
Y_ALL = df_unificado['n_siniestros'].to_numpy(float)
OFF_ALL = np.log(df_unificado['long_seg_m'].to_numpy(float))

## Covariables: log1p en distancias (se estandarizan por pliegue) y dummies de velocidad reglamentaria
X0 = df_unificado[COL_DIST + ['MVINUMC']].copy()
X0[COL_DIST] = np.log1p(X0[COL_DIST])
DUM = pd.get_dummies(df_unificado['MVI_VELREG'].astype(int).astype(str), prefix='vel', drop_first=True, dtype=float)


def armar_disenio(pos_tr, pos_te):
    ## Matriz del conteo (X) y de la inflación (Xi = constante + log-longitud estandarizado); el escalador usa solo entrenamiento
    esc = StandardScaler().fit(X0.iloc[pos_tr][COL_DIST])
    mu_l, sd_l = OFF_ALL[pos_tr].mean(), OFF_ALL[pos_tr].std()

    def _X(pos):
        X = X0.iloc[pos].copy()
        X[COL_DIST] = esc.transform(X[COL_DIST])
        return sm.add_constant(pd.concat([X, DUM.iloc[pos]], axis=1), has_constant='add')

    def _Xi(pos, X):
        return pd.DataFrame({'const': 1.0, 'log_long_z': (OFF_ALL[pos] - mu_l) / sd_l}, index=X.index)

    X_tr, X_te = _X(pos_tr), _X(pos_te)
    return dict(X_tr=X_tr, X_te=X_te, Xi_tr=_Xi(pos_tr, X_tr), Xi_te=_Xi(pos_te, X_te),
                off_tr=OFF_ALL[pos_tr], off_te=OFF_ALL[pos_te], y_tr=Y_ALL[pos_tr], y_te=Y_ALL[pos_te])

## 4. Modelos candidatos

In [6]:
## Ajuste robusto: prueba optimizadores en cadena con arranque en caliente; maxiter explícito (el defecto de statsmodels, 35, no alcanza)
def _num(a):
    return np.asarray(a, dtype=float)


def _fit_robusto(modelo, start=None, plan=(('bfgs', 500), ('lbfgs', 1000), ('nm', 4000))):
    mejor, ok = None, False
    for metodo, it in plan:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                r = modelo.fit(start_params=start, method=metodo, maxiter=it, disp=0)
            except Exception:
                continue
        p = _num(r.params)
        if not np.all(np.isfinite(p)):
            continue
        conv = bool(r.mle_retvals.get('converged', False))
        if mejor is None or conv or r.llf > mejor.llf:
            mejor, ok = r, conv
        if conv:
            break
        start = p
    if mejor is None:
        raise RuntimeError('Ningún optimizador devolvió parámetros finitos')
    return mejor, ok

In [7]:
## Los cuatro modelos con la misma interfaz: media, logpmf, parámetros, AIC, BIC y convergencia
def ajustar(tipo, y, X, off, Xi=None, params_bn=None):
    y, off, res, alpha, pi = _num(y), _num(off), None, np.nan, None

    if tipo == 'nulo':
        b0 = np.log(y.sum() / np.exp(off).sum())    # MLE cerrado de Poisson solo con intercepto y offset
        media = lambda X, off, Xi=None: np.exp(b0 + _num(off))
        logpmf = lambda y, X, off, Xi=None: stats.poisson.logpmf(_num(y), media(X, off))
        params, ok = np.array([b0]), True

    elif tipo == 'poisson':
        res = sm.GLM(y, X, family=sm.families.Poisson(), offset=off).fit()
        params, ok = _num(res.params), bool(res.converged)
        media = lambda X, off, Xi=None: np.exp(np.clip(_num(X) @ params + _num(off), -30, 30))
        logpmf = lambda y, X, off, Xi=None: stats.poisson.logpmf(_num(y), media(X, off))

    elif tipo == 'bn':
        res, ok = _fit_robusto(sm.NegativeBinomial(y, X, offset=off, loglike_method='nb2'))
        params = _num(res.params)
        b, alpha = params[:-1], max(params[-1], 1e-8)
        if params[-1] < 1e-4:
            warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)
        media = lambda X, off, Xi=None: np.exp(np.clip(_num(X) @ b + _num(off), -30, 30))
        # NB2: n = 1/alpha y p = 1/(1 + alpha*mu) dan media mu y varianza mu + alpha*mu^2
        logpmf = lambda y, X, off, Xi=None: stats.nbinom.logpmf(_num(y), 1 / alpha, 1 / (1 + alpha * media(X, off)))

    else:
        k_inf = Xi.shape[1]
        mod = ZeroInflatedNegativeBinomialP(y, X, exog_infl=Xi, offset=off, inflation='logit', p=2)
        start = np.r_[-3.0, np.zeros(k_inf - 1), _num(params_bn)] if params_bn is not None else None
        res, ok = _fit_robusto(mod, start)
        params = _num(res.params)                      # orden de statsmodels: [inflación, conteo, alpha]
        g, b, alpha = params[:k_inf], params[k_inf:-1], max(params[-1], 1e-8)
        mu = lambda X, off: np.exp(np.clip(_num(X) @ b + _num(off), -30, 30))
        pi = lambda Xi: expit(_num(Xi) @ g)
        media = lambda X, off, Xi: (1 - pi(Xi)) * mu(X, off)

        def logpmf(y, X, off, Xi):
            y_, m_, p_ = _num(y), mu(X, off), pi(Xi)
            lnb = stats.nbinom.logpmf(y_, 1 / alpha, 1 / (1 + alpha * m_))
            with np.errstate(divide='ignore'):
                return np.where(y_ == 0, np.logaddexp(np.log(p_), np.log1p(-p_) + lnb), np.log1p(-p_) + lnb)

    n, k = len(y), len(params)
    llf = float(logpmf(y, X, off, Xi).sum())
    if res is not None and abs(res.llf - llf) > 1e-3 * abs(res.llf) + 1e-2:
        warnings.warn(f'{tipo}: llf propio ({llf:.2f}) != llf de statsmodels ({res.llf:.2f})', RuntimeWarning)
    return dict(tipo=tipo, res=res, media=media, logpmf=logpmf, params=params, k=k, llf=llf,
                aic=-2 * llf + 2 * k, bic=-2 * llf + k * np.log(n), convergio=ok, alpha=alpha, pi=pi)


def ajustar_todos(D):
    ## El ZINB arranca desde los parámetros de la BN
    args = (D['y_tr'], D['X_tr'], D['off_tr'])
    m = {'Nulo': ajustar('nulo', *args), 'Poisson': ajustar('poisson', *args), 'BN': ajustar('bn', *args)}
    m['ZINB'] = ajustar('zinb', *args, D['Xi_tr'], params_bn=m['BN']['params'])
    return m


def metricas(m, y, X, off, Xi=None):
    y = _num(y)
    mu = np.clip(m['media'](X, off, Xi), 1e-9, None)
    return dict(MAE=mean_absolute_error(y, mu), RMSE=float(np.sqrt(mean_squared_error(y, mu))),
                devianza=mean_poisson_deviance(y, mu), logscore=float(m['logpmf'](y, X, off, Xi).mean()))

## 5. Ajuste sobre todos los segmentos y selección entre Poisson y BN

In [8]:
## Ajuste con todos los segmentos; el ZINB usa inflación logit(pi) = g0 + g1*log-longitud estandarizada
pos_all = np.arange(len(df_unificado))
D = armar_disenio(pos_all, pos_all)
aj = ajustar_todos(D)

comp = pd.DataFrame([dict(modelo=nom, parametros=m['k'], logLik=m['llf'], AIC=m['aic'], BIC=m['bic'], alpha=m['alpha'],
                          pi_medio=float(m['pi'](D['Xi_tr']).mean()) if m['pi'] else np.nan, convergio=m['convergio'])
                     for nom, m in aj.items()]).set_index('modelo')
print(comp.round(4).to_string())

         parametros      logLik          AIC          BIC   alpha  pi_medio  convergio
modelo                                                                                
Nulo              1 -63977.1656  127956.3312  127966.1195     NaN       NaN       True
Poisson          11 -58622.7360  117267.4720  117375.1435     NaN       NaN       True
BN               12 -52149.2038  104322.4076  104439.8675  3.8353       NaN       True
ZINB             14 -52002.4847  104032.9694  104170.0059  2.6059    0.1499       True


In [9]:
## Poisson vs BN: razón de verosimilitud sobre alpha con p-valor corregido por la frontera (alpha = 0)
LR = max(2 * (aj['BN']['llf'] - aj['Poisson']['llf']), 0)
P_LR = 0.5 * stats.chi2.sf(LR, 1)    # mezcla 50:50 de chi2(0) y chi2(1), por eso la mitad del p-valor convencional
BASE = 'BN' if P_LR < 0.05 else 'Poisson'

print(f"LR = {LR:.2f} | p (corregido por frontera) = {P_LR:.3g} | alpha BN = {aj['BN']['alpha']:.4f}")
print(f"Modelo base: {BASE}")

LR = 12947.06 | p (corregido por frontera) = 0 | alpha BN = 3.8353
Modelo base: BN


## 6. Validación cruzada aleatoria (K = 5)

In [10]:
## Cinco particiones 80/20 con validaciones disjuntas; cada pliegue reajusta los cuatro modelos
filas = []
for f, (pos_tr, pos_te) in enumerate(KFold(K_PLIEGUES, shuffle=True, random_state=SEMILLA).split(pos_all)):
    Df = armar_disenio(pos_tr, pos_te)
    mods = ajustar_todos(Df)
    dev_nulo = metricas(mods['Nulo'], Df['y_te'], Df['X_te'], Df['off_te'])['devianza']
    for nom, m in mods.items():
        r = metricas(m, Df['y_te'], Df['X_te'], Df['off_te'], Df['Xi_te'])
        filas.append(dict(r, fold=f, modelo=nom, convergio=m['convergio'], dev_explicada=1 - r['devianza'] / dev_nulo))
    print(f'Pliegue {f + 1}/{K_PLIEGUES} listo')

res_cv = pd.DataFrame(filas)

Pliegue 1/5 listo
Pliegue 2/5 listo
Pliegue 3/5 listo
Pliegue 4/5 listo
Pliegue 5/5 listo


In [11]:
## Media y desviación estándar entre pliegues (menor es mejor: MAE, RMSE, devianza; mayor es mejor: logscore, dev_explicada)
metr = ['MAE', 'RMSE', 'devianza', 'logscore', 'dev_explicada']
tabla_cv = res_cv.groupby('modelo')[metr].agg(['mean', 'std']).reindex(['Nulo', 'Poisson', 'BN', 'ZINB'])
print(tabla_cv.round(4).to_string())
print('Ajustes que no convergieron en la validación:', int((~res_cv['convergio']).sum()))

            MAE            RMSE         devianza         logscore         dev_explicada        
           mean     std    mean     std     mean     std     mean     std          mean     std
modelo                                                                                         
Nulo     0.2654  0.0020  0.6058  0.0352   0.7460  0.0119  -0.4858  0.0057        0.0000  0.0000
Poisson  0.2478  0.0022  0.5990  0.0346   0.6651  0.0104  -0.4453  0.0057        0.1084  0.0073
BN       0.2655  0.0029  0.6312  0.0311   0.6746  0.0108  -0.3961  0.0048        0.0957  0.0111
ZINB     0.2584  0.0023  0.6067  0.0334   0.6580  0.0106  -0.3950  0.0047        0.1179  0.0099
Ajustes que no convergieron en la validación: 0


## 7. Decisión del modelo final

In [12]:
## El ZINB reemplaza a la BN solo si mejora el BIC, el log-score y la devianza fuera de muestra, y si BN y ZINB convergen en todos los ajustes
piv = res_cv.pivot(index='fold', columns='modelo', values=['logscore', 'devianza'])
d_ls = float((piv['logscore']['ZINB'] - piv['logscore']['BN']).mean())
d_dv = float((piv['devianza']['BN'] - piv['devianza']['ZINB']).mean())
crit_bic = aj['ZINB']['bic'] < aj['BN']['bic']
crit_cv = d_ls > 0 and d_dv > 0
conv_ok = all(aj[n]['convergio'] for n in ('BN', 'ZINB')) and bool(res_cv.loc[res_cv['modelo'].isin(['BN', 'ZINB']), 'convergio'].all())

MODELO_FINAL = 'ZINB' if (BASE == 'BN' and crit_bic and crit_cv and conv_ok) else BASE

print(f"BIC BN / ZINB: {aj['BN']['bic']:.1f} / {aj['ZINB']['bic']:.1f} -> favorece ZINB: {crit_bic}")
print(f"CV: mejora media de logscore = {d_ls:+.5f} | de devianza = {d_dv:+.5f} -> favorece ZINB: {crit_cv}")
print(f"BN y ZINB convergieron en todos los ajustes: {conv_ok}")
print(f"Modelo final: {MODELO_FINAL}")

BIC BN / ZINB: 104439.9 / 104170.0 -> favorece ZINB: True
CV: mejora media de logscore = +0.00111 | de devianza = +0.01656 -> favorece ZINB: True
BN y ZINB convergieron en todos los ajustes: True
Modelo final: ZINB


In [13]:
## Coeficientes del modelo final ajustado con todos los segmentos: exp(coef) es IRR en el conteo y OR en la inflación
res_f = aj[MODELO_FINAL]['res']
tab = pd.DataFrame({'coef': _num(res_f.params), 'ee': _num(res_f.bse), 'p': _num(res_f.pvalues)}, index=res_f.params.index)
tab['exp(coef)'] = np.where(tab.index == 'alpha', np.nan, np.exp(tab['coef']))
print(tab.round(4).to_string())

                           coef      ee       p  exp(coef)
inflate_const           -1.8959  0.1297  0.0000     0.1502
inflate_log_long_z       0.7052  0.0460  0.0000     2.0243
const                   -6.9682  0.0381  0.0000     0.0009
dist_puente_peatonal_m  -0.0053  0.0101  0.5971     0.9947
dist_puente_vehicular_m  0.0697  0.0102  0.0000     1.0722
dist_semaforo_m         -0.3786  0.0093  0.0000     0.6848
dist_peatonal_m         -0.0414  0.0119  0.0005     0.9595
MVINUMC                  0.3958  0.0156  0.0000     1.4856
vel_36                   0.7005  0.0587  0.0000     2.0148
vel_42                   0.9171  0.0846  0.0000     2.5020
vel_48                   0.9372  0.0986  0.0000     2.5528
vel_54                   0.6698  0.1144  0.0000     1.9539
vel_60                   0.9795  0.0246  0.0000     2.6631
alpha                    2.6059  0.0970  0.0000        NaN


# Resultados
